# Tiled CPU grid backend for the lean ant: what it saves and what it costs

**Question.** `sweep_lean`'s dense grid writes `(size+2)^2` bytes per worker per rule. Does a tiled grid (sparse 64x64 tiles; bounded cache; packed or LZ4-compressed inactive tiles) shrink the process at an acceptable throughput cost, with *identical* output?

**Assumptions.** Same bounded board and escape semantics as dense; workloads are five census rules used as workloads, not as subjects (no scientific claim about them); one machine (WSL2, Xeon W-2125), single thread, shared with another contributor; dense throughput is noisy to ~20%.

Predictions and adoption criteria were registered in `README.md` before timing; this notebook only reads the committed `results/` and agrees with the README's Results section.

In [1]:
from pathlib import Path
import json, polars as pl
res = Path('..') / 'results' if Path('../results').exists() else Path('results')
prov = (res / 'runs_provenance.txt').read_text().splitlines()
print('\n'.join(l for l in prov if l.startswith(('date','uname','cpu','rustc','bench commit','loadavg')) or 'sweep_lean' in l))
print((res / 'runs_d_provenance.txt').read_text().splitlines()[0:9][-3:])

date: 2026-10-23T19:00:44+13:00
uname: Linux 6.18.54.1-microsoft-standard-WSL2 #1 SMP PREEMPT_DYNAMIC Tue Oct  6 19:13:07 UTC 2026 x86_64
cpu: Model name: Intel(R) Xeon(R) W-2125 CPU @ 4.00GHz
rustc: rustc 1.98.1 (48a229cea 2026-09-01)  cargo: cargo 1.98.1 (797e8a9bc 2026-08-05)
bench commit: e93d4c7c18a428e07f2a7f1621ca27798fd1f0d2
43ee4e00aafb4662d8ccdf0f7eca121cae3f36fd7183d51de3cc8bc3035a9a9c  /home/tim/bench-e93d4c7c18a428e07f2a7f1621ca27798fd1f0d2/target/release/sweep_lean
loadavg at start: 0.72 0.15 0.05 2/323 3010
loadavg at end: 1.00 0.65 0.29 1/297 4279
['sha256:', '2cb314a0c635eb122bf81822f401e8804879182b44e1a4efe855ec5ba76b472e  /home/tim/bench-b53bcb69d15c24908805ea374e2e05118bdfe648/target/release/sweep_lean', '860e4e5cef9f4dec9991ea19d2eb10819e055cbe17fde41eeb4e8844d1671a89  /home/tim/bench-b53bcb69d15c24908805ea374e2e05118bdfe648/target/release/examples/tiled_grid_backend_codec']


## Exactness gate
Every benchmark run's CSV was compared byte-for-byte with the dense run of the same repetition.

In [2]:
ident = pl.concat([pl.read_csv(res / d / 'identity.tsv', separator='\t', has_header=False, new_columns=['wl','cfg','rep','same']) for d in ('runs','runs_d')])
print(ident.height, 'runs;', ident.filter(pl.col('same') != 1).height, 'differ from dense')

204 runs; 0 differ from dense


## Memory and throughput (medians over repetitions)
Ratios against dense are within noise unless they differ from 1 by more than ~1.2x.

In [3]:
a = pl.read_csv(res / 'summary.csv')
cols = ['wl','cfg','reps','sps_med','tput_vs_dense','rss_med','rss_vs_dense','rss_vs_B6','transitions_per_step','misses','evictions','end_tiles']
with pl.Config(tbl_rows=60, tbl_cols=20, tbl_width_chars=200):
    print(a.select(cols).with_columns(pl.col(pl.Float64).round(3)))

shape: (31, 12)
┌─────┬───────┬──────┬────────────┬───────────────┬──────────┬──────────────┬───────────┬──────────────────────┬─────────┬───────────┬───────────┐
│ wl  ┆ cfg   ┆ reps ┆ sps_med    ┆ tput_vs_dense ┆ rss_med  ┆ rss_vs_dense ┆ rss_vs_B6 ┆ transitions_per_step ┆ misses  ┆ evictions ┆ end_tiles │
│ --- ┆ ---   ┆ ---  ┆ ---        ┆ ---           ┆ ---      ┆ ---          ┆ ---       ┆ ---                  ┆ ---     ┆ ---       ┆ ---       │
│ str ┆ str   ┆ i64  ┆ f64        ┆ f64           ┆ f64      ┆ f64          ┆ f64       ┆ f64                  ┆ i64     ┆ i64       ┆ i64       │
╞═════╪═══════╪══════╪════════════╪═══════════════╪══════════╪══════════════╪═══════════╪══════════════════════╪═════════╪═══════════╪═══════════╡
│ W1  ┆ A     ┆ 5    ┆ 2.1692e8   ┆ 1.0           ┆ 70.066   ┆ 1.0          ┆ 7.923     ┆ null                 ┆ null    ┆ null      ┆ null      │
│ W1  ┆ B5    ┆ 5    ┆ 2.0921e8   ┆ 0.964         ┆ 8.91     ┆ 0.127        ┆ 1.008     ┆ 0.031       

## Stage D (LZ4 on packed tiles), separate batch with its own dense/B6/C1024 references

In [4]:
d = pl.read_csv(res / 'summary_runs_d.csv')
with pl.Config(tbl_rows=60, tbl_cols=20, tbl_width_chars=200):
    print(d.select(cols).with_columns(pl.col(pl.Float64).round(3)))

shape: (11, 12)
┌─────┬───────┬──────┬──────────┬───────────────┬─────────┬──────────────┬───────────┬──────────────────────┬────────┬───────────┬───────────┐
│ wl  ┆ cfg   ┆ reps ┆ sps_med  ┆ tput_vs_dense ┆ rss_med ┆ rss_vs_dense ┆ rss_vs_B6 ┆ transitions_per_step ┆ misses ┆ evictions ┆ end_tiles │
│ --- ┆ ---   ┆ ---  ┆ ---      ┆ ---           ┆ ---     ┆ ---          ┆ ---       ┆ ---                  ┆ ---    ┆ ---       ┆ ---       │
│ str ┆ str   ┆ i64  ┆ f64      ┆ f64           ┆ f64     ┆ f64          ┆ f64       ┆ f64                  ┆ i64    ┆ i64       ┆ i64       │
╞═════╪═══════╪══════╪══════════╪═══════════════╪═════════╪══════════════╪═══════════╪══════════════════════╪════════╪═══════════╪═══════════╡
│ W1  ┆ A     ┆ 5    ┆ 2.4213e8 ┆ 1.0           ┆ 70.164  ┆ 1.0          ┆ 7.847     ┆ null                 ┆ null   ┆ null      ┆ null      │
│ W1  ┆ B6    ┆ 5    ┆ 2.8249e8 ┆ 1.167         ┆ 8.941   ┆ 0.127        ┆ 1.0       ┆ 0.016                ┆ 0      ┆ 0      

## Would block compression pay? (criterion A5, per-tile deflate/xz on real canonical payloads)

In [5]:
ce = json.loads((res / 'compress_estimate.json').read_text())
for r in ce: print(r['rule'], 'tiles', r['tiles'], 'packed', r['packed_total_bytes'], 'deflate9 x', r['deflate9']['ratio_vs_packed'], 'xz6 x', r['xz6']['ratio_vs_packed'])
print(open(res / 'codec_bench.tsv').read())

RLR tiles 216 packed 331776 deflate9 x 1.947 xz6 x 1.741
RRRRRRRLLRRR tiles 4 packed 10240 deflate9 x 28.132 xz6 x 28.055
RLLRLLLRRRLL tiles 12544 packed 32112640 deflate9 x 80.455 xz6 x 73.995
rule	n	w	packed_bytes	raw_bytes	encode_us	decode_us	cell_ns_encode	cell_ns_decode
RLR	3	2	1536	4096	13.06	14.86	3.19	3.63
RRRRRRRLLRRR	12	4	2560	4096	10.35	14.30	2.53	3.49
RLLRLLLRRRLL	12	4	2560	4096	11.64	14.23	2.84	3.47



## What this does and does not show
* Exact: all benchmark runs are byte-identical to dense; the oracle tests compare whole states.
* Sparse tiles: ~0.1x dense RSS on compact/thin workloads, 0.003x on a 65536 board; 12% on a pattern that fills the board.
* Packing alone saves 33% on the large structured footprint with a big cache and thrashes with a small one.
* LZ4 gives 0.19x dense on the structured W3 only (one rule); chaotic W1 compresses ~2x and is small anyway.
* Not shown: multi-thread behaviour, other CPUs, chaotic rules at 1e10 steps, tile-offset effects, probe memory.

**Caveats.** Dense is noisy to ~20% in this shared environment; 'tiled is faster than dense' is weak evidence and an untested cache-layout hypothesis. **Next steps:** tiled probe; tile-offset ablation for compact starts.